In [17]:
import httpx
import time
import re
import pandas as pd

In [6]:
client = httpx.Client(
    timeout=httpx.Timeout(connect=5.0, read=15.0, write=15.0, pool=15.0),
    limits=httpx.Limits(max_keepalive_connections=0, max_connections=1),
    trust_env=False,
    follow_redirects=True,
    headers={
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "zh-CN,zh;q=0.9",
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0 Safari/537.36"
        ),
    },
)

SINA_REFERER_HEADERS = {"Referer": "https://vip.stock.finance.sina.com.cn/mkt/"}


### 新浪行情中心
对应地址 - https://vip.stock.finance.sina.com.cn/mkt/

该行情地址通过`https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeData?page=1&num=40&sort=symbol&asc=1&node=sh_a&symbol=&_s_r_a=init`（GET）请求行情数据。`page`和`num`分别控制页码和每页数量，`sort`控制排序字段，与`asc`配合控制排序方向（`1`升序、`0`降序）；`node`参数尤其重要，用于筛选不同分类的行情数据。`symbol`通常传空字符串，`_s_r_a`是新浪页面内部参数，有点像request action，比如page代表是翻页触发的请求，init是页面第一次加载等等。

`node`实际的取值范围可参照`https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodes`。`hs_a`、`sh_a`、`sz_a`、`hs_bjs`、`cyb`、`kcb`就分别对应沪深 A 股整体、沪 A、深 A、北交所、创业板、科创板，构建 A 股标的池时优先使用这些节点。`hs_s`表示沪深指数集合，`etf_hq_fund` 适合获取场内 ETF 基金行情，是构建 ETF 标的池的首选节点，同时 LOF 基金使用`lof_hq_fund`。

对于接口调用的response数据中，symbol 是带交易所前缀的证券标识（如 bj920000、sh510010），code 是六位证券代码，name 是名称，trade 是最新成交价，pricechange 是相对昨收的涨跌额，changepercent 是涨跌幅（单位为 %），buy 和 sell 分别是当前最优买价和卖价，settlement 是昨收/前一结算价，open、high、low 分别是今开、最高价和最低价，volume 是成交量（股票为股、ETF 为份额），amount 是成交额（元），ticktime 是行情更新时间，per 是市盈率，pb 是市净率；比如安徽凤凰作为股票，这两个字段分别为 18.833 和 1.82，而 180 治理 ETF 返回 0，通常表示不适用或未提供，mktcap 是总市值、nmc 是流通市值，按该接口口径通常以万元计，turnoverratio 是换手率（单位为 %）。

In [7]:
HQ_ENDPOINT = "https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeData"
HQ_COUNT_ENDPOINT = "https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeStockCount"

In [12]:
HS_A_PARAMS={
    "page": 1,
    "num": 10,
    "sort": "symbol",
    "asc": 1,
    "node": 'hs_a',
    "symbol": "",
    "_s_r_a": "page",
}

HS_A_COUNT_PARAMS={
    "node": 'hs_a',
}

hs_a_counts = int(client.request(method="GET", url=HQ_COUNT_ENDPOINT, params=HS_A_COUNT_PARAMS, headers=SINA_REFERER_HEADERS).json())

ha_s_resp = client.request(method="GET", url=HQ_ENDPOINT, params=HS_A_PARAMS, headers=SINA_REFERER_HEADERS)
ha_s_resp_json = ha_s_resp.json()
ha_s_resp_json[0]

{'symbol': 'bj920000',
 'code': '920000',
 'name': '安徽凤凰',
 'trade': '13.560',
 'pricechange': 0.14,
 'changepercent': 1.043,
 'buy': '13.460',
 'sell': '13.560',
 'settlement': '13.420',
 'open': '13.380',
 'high': '13.740',
 'low': '13.000',
 'volume': 656429,
 'amount': 8895246,
 'ticktime': '15:30:01',
 'per': 18.833,
 'pb': 1.82,
 'mktcap': 124318.08,
 'nmc': 78097.3623,
 'turnoverratio': 1.13975}

In [13]:
ETF_PARAMS={
    "page": 1,
    "num": 10,
    "sort": "symbol",
    "asc": 1,
    "node": 'etf_hq_fund',
    "symbol": "",
    "_s_r_a": "page",
}

ETF_COUNT_PARAMS={
    "node": 'etf_hq_fund',
}

etf_counts = int(client.request(method="GET", url=HQ_COUNT_ENDPOINT, params=ETF_COUNT_PARAMS, headers=SINA_REFERER_HEADERS).json())
print(f"ETF counts: {etf_counts}")

etf_resp = client.request(method="GET", url=HQ_ENDPOINT, params=ETF_PARAMS, headers=SINA_REFERER_HEADERS)
etf_resp_json = etf_resp.json()
etf_resp_json[0]

ETF counts: 1639


{'symbol': 'sh510010',
 'code': '510010',
 'name': '180治理ETF交银',
 'trade': '1.756',
 'pricechange': 0.002,
 'changepercent': 0.114,
 'buy': '1.756',
 'sell': '1.762',
 'settlement': '1.754',
 'open': '1.762',
 'high': '1.766',
 'low': '1.749',
 'volume': 41300,
 'amount': 72833,
 'ticktime': '15:00:03',
 'per': 0,
 'pb': 0,
 'mktcap': 23271.2779672,
 'nmc': 22920.08464,
 'turnoverratio': 0.03164}

In [14]:
HQ_SIMPLE_ENDPOINT = "https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeDataSimple"
HQ_SIMPLE_COUNT_ENDPOINT = "https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeStockCountSimple"

etf_simple_counts = int(client.request(method="GET", url=HQ_SIMPLE_COUNT_ENDPOINT, params=ETF_COUNT_PARAMS, headers=SINA_REFERER_HEADERS).json())
print(f"ETF counts: {etf_simple_counts}")

etf_simple_resp = client.request(method="GET", url=HQ_SIMPLE_ENDPOINT, params=ETF_PARAMS, headers=SINA_REFERER_HEADERS)
etf_simple_resp_json = etf_simple_resp.json()
etf_simple_resp_json[0]

ETF counts: 1640


{'symbol': 'sh510010',
 'name': '180治理ETF交银',
 'trade': '1.756',
 'pricechange': '0.002',
 'changepercent': '0.114',
 'buy': '1.756',
 'sell': '1.762',
 'settlement': '1.754',
 'open': '1.762',
 'high': '1.766',
 'low': '1.749',
 'volume': 41300,
 'amount': 72833,
 'code': '510010',
 'ticktime': '15:00:03',
 'state': '00',
 'statetxt': '正常'}

### 单个标的行情快照

In [25]:
SINGLE_QUOTE_ENDPOINT = "http://hq.sinajs.cn/rn={timestamp}&list={symbol}"
SINGLE_QUOTE_HEADER = {"Referer": "http://finance.sina.com.cn/"}

sina_symbol = "sh515080"
sina_quote_resp = client.get(
    SINGLE_QUOTE_ENDPOINT.format(timestamp=int(time.time()), symbol=sina_symbol),
    headers= SINGLE_QUOTE_HEADER
)

# response header 是 Content-Type: application/javascript; charset=GB18030
print(f"{sina_quote_resp.headers}")
# 所以不是取json()，而是取 text
# 'var hq_str_sh513190="港股通金融ETF华夏,1.800,1.800,1.822,1.833,1.799,1.822,1.823,169558700,309415609.000,76200,1.822,40500,1.821,39500,1.820,95600,1.819,9900,1.818,114200,1.823,651900,1.824,37000,1.825,107500,1.826,42000,1.827,2026-08-26,11:30:00,00,";\n'
# .匹配任意符号，*匹配任意数量,?非贪婪匹配，尽可能少匹配
pattern = rf'hq_str_{sina_symbol}="(.*?)";'
sina_quote_match = re.search(pattern, sina_quote_resp.text)
print((f"Raw Text: {sina_quote_resp.text}"))
print(f"Full Match: {sina_quote_match.group(0)}")
print(f"Quote Snapshot: {sina_quote_match.group(1)}")
# name，今开，昨收，最新价，今日最高，今日最低，买一，卖一，成交量，成交额，买一量，买一价，买二量，买二价，买三量，买三价，买四量，买四价，买五量，买五价，卖一量，卖一价，卖二量，卖二价，卖三量，卖三价，卖四量，卖四价，卖五量，卖五价，日期，时间

Headers({'cache-control': 'no-cache', 'content-length': '185', 'connection': 'Keep-Alive', 'content-type': 'application/javascript; charset=GB18030', 'content-encoding': 'gzip'})
Raw Text: var hq_str_sh515080="中证红利ETF招商,1.590,1.594,1.605,1.607,1.585,1.604,1.605,154555100,247375157.000,246400,1.604,2320600,1.603,3121700,1.602,3224200,1.601,3722400,1.600,2544300,1.605,3295500,1.606,4864800,1.607,7573100,1.608,1598400,1.609,2026-08-26,13:25:48,00,";

Full Match: hq_str_sh515080="中证红利ETF招商,1.590,1.594,1.605,1.607,1.585,1.604,1.605,154555100,247375157.000,246400,1.604,2320600,1.603,3121700,1.602,3224200,1.601,3722400,1.600,2544300,1.605,3295500,1.606,4864800,1.607,7573100,1.608,1598400,1.609,2026-08-26,13:25:48,00,";
Quote Snapshot: 中证红利ETF招商,1.590,1.594,1.605,1.607,1.585,1.604,1.605,154555100,247375157.000,246400,1.604,2320600,1.603,3121700,1.602,3224200,1.601,3722400,1.600,2544300,1.605,3295500,1.606,4864800,1.607,7573100,1.608,1598400,1.609,2026-08-26,13:25:48,00,


### K 线数据接口
对应地址 - "https://money.finance.sina.com.cn/quotes_service/api/json_v2.php/CN_MarketData.getKLineData"

symbol 参数控制查询标的的代码，交易所前缀配合6位代码；scale指示查询的 K 线周期，比如，5分钟用5，15分钟用15，30分钟够用30，60分钟用60，日线用240，周线1200，月线7200；ma指定是否返回移动平均线，如果不需要就是no，否则传入需要的周期整数，特殊的如果需要多均线用“5,10,30"的写法。最终 datalen指定返回需要多少根K线，结果从旧到新排列。

In [27]:
KLINE_ENDPOINT = "https://money.finance.sina.com.cn/quotes_service/api/json_v2.php/CN_MarketData.getKLineData"
KLINE_HEADERS = {"Referer": "https://finance.sina.com.cn/"}

kline_symbol = "sh515080"
kline_date = "2026-08-25"
kline_resp = client.get(
    KLINE_ENDPOINT,
    params={
        "symbol": kline_symbol,
        "scale": 240,
        "ma": "no",
        "datalen": 100,
    },
    headers=KLINE_HEADERS,
)

kline_rows = kline_resp.json()
kline_rows


[{'day': '2026-04-01',
  'open': '1.605',
  'high': '1.611',
  'low': '1.597',
  'close': '1.602',
  'volume': '166833135'},
 {'day': '2026-04-02',
  'open': '1.602',
  'high': '1.610',
  'low': '1.597',
  'close': '1.604',
  'volume': '141836376'},
 {'day': '2026-04-03',
  'open': '1.603',
  'high': '1.604',
  'low': '1.574',
  'close': '1.580',
  'volume': '265010326'},
 {'day': '2026-04-07',
  'open': '1.581',
  'high': '1.586',
  'low': '1.572',
  'close': '1.586',
  'volume': '202932465'},
 {'day': '2026-04-08',
  'open': '1.584',
  'high': '1.598',
  'low': '1.576',
  'close': '1.597',
  'volume': '221522199'},
 {'day': '2026-04-09',
  'open': '1.592',
  'high': '1.596',
  'low': '1.583',
  'close': '1.587',
  'volume': '156184000'},
 {'day': '2026-04-10',
  'open': '1.587',
  'high': '1.596',
  'low': '1.584',
  'close': '1.589',
  'volume': '164489400'},
 {'day': '2026-04-13',
  'open': '1.588',
  'high': '1.592',
  'low': '1.580',
  'close': '1.587',
  'volume': '130481200'},
